#### Import Packages

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor

##### Import cleaned df

In [ ]:
file_path = "cleaned_df_4lakh.csv"
cleaned_df = pd.read_csv(file_path)
print("flat_df loaded successfully.")
print("First 5 rows:")
display(cleaned_df.tail())

In [ ]:
X_cols = ['lon', 'lat', 'elevation', 'ph', 'organic_matter (%)',
          'total_nitrogen (%)', 'potassium (kg/ha)', 'p2o5 (kg/ha)',
          'boron (ppm)', 'zinc (ppm)', 'sand (%)', 'clay (%)', 'slit (%)',
          'parentsoil']

In [ ]:
Y_cols = ['Maize_OPV_MOP', 'Maize_OPV_DAP', 'Maize_OPV_UREA1', 'Maize_OPV_UREA2',
          'Maize_OPV_UREA3', 'Maize_OPV_organic', 'Maize_OPV_zinc', 'Maize_OPV_boron',
          'Maize_Hybrid_MOP', 'Maize_Hybrid_DAP', 'Maize_Hybrid_UREA1',
          'Maize_Hybrid_UREA2', 'Maize_Hybrid_UREA3', 'Maize_Hybrid_organic',
          'Maize_Hybrid_zinc', 'Maize_Hybrid_boron', 'Rice_OPV_MOP',
          'Rice_OPV_DAP', 'Rice_OPV_UREA1', 'Rice_OPV_UREA2', 'Rice_OPV_UREA3',
          'Rice_OPV_organic', 'Rice_OPV_zinc', 'Rice_OPV_boron', 'Rice_Hybrid_MOP',
          'Rice_Hybrid_DAP', 'Rice_Hybrid_UREA1', 'Rice_Hybrid_UREA2',
          'Rice_Hybrid_UREA3', 'Rice_Hybrid_organic', 'Rice_Hybrid_zinc',
          'Rice_Hybrid_boron', 'Wheat_OPV_MOP', 'Wheat_OPV_DAP', 'Wheat_OPV_UREA1',
          'Wheat_OPV_UREA2', 'Wheat_OPV_UREA3', 'Wheat_OPV_organic',
          'Wheat_OPV_zinc', 'Wheat_OPV_boron']

In [ ]:
X = cleaned_df[X_cols]
Y = cleaned_df[Y_cols]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
import logging
from datetime import datetime

logging.basicConfig(
    filename=f"training_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

def log(msg):
    print(msg)
    logging.info(msg)

In [ ]:
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

models = {
    "Linear Regression": MultiOutputRegressor(
        LinearRegression()
    ),

    "Ridge Regression": MultiOutputRegressor(
        Ridge()
    ),

    "XGBoost": MultiOutputRegressor(
        XGBRegressor(
            random_state=42,
            n_jobs=1
        )
    ),

    "LightGBM": MultiOutputRegressor(
        LGBMRegressor(
            random_state=42,
            n_jobs=1,
            verbose=-1
        )
    ),
    "Random Forest": MultiOutputRegressor(
        RandomForestRegressor(
            random_state=42,
            n_jobs=1
        )
    )
}

In [ ]:
param_grids = {
    "Linear Regression": {},

    "Ridge Regression": {
        "model__estimator__alpha": [0.01, 0.1, 1, 10, 100]
    },

    "Random Forest": {
        "model__estimator__n_estimators": [20, 100, 200, 300],
        "model__estimator__max_depth": [5, 10, 20, None],
        "model__estimator__min_samples_split": [2, 5]
    },

    "XGBoost": {
        "model__estimator__n_estimators": [100, 300, 500],
        "model__estimator__learning_rate": [0.01, 0.05, 0.1],
        "model__estimator__max_depth": [3, 5, 7],
        "model__estimator__subsample": [0.8, 1.0]
    },

    "LightGBM": {
        "model__estimator__n_estimators": [100, 300, 500],
        "model__estimator__learning_rate": [0.01, 0.05, 0.1],
        "model__estimator__num_leaves": [15, 31, 63],
        "model__estimator__min_data_in_leaf": [10, 20, 50]
    }
}

In [ ]:
from sklearn.pipeline import Pipeline
from time import time
import joblib

In [ ]:
best_models = {}

for name, model in models.items():

    print(f"\n🔍 Training/Tuning {name}...")

    start = time()

    pipe = Pipeline([
        ("model", model)
    ])

    params = param_grids[name]

    if params == {}:

        pipe.fit(X_train, y_train)

        best_models[name] = pipe

        elapsed = time() - start

        print("✅ No tuning needed.")
        print(f"Training Time: {elapsed:.2f} sec")

        continue

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=params,
        n_iter=10,
        scoring="r2",
        cv=3,
        verbose=3,
        random_state=42,
        n_jobs=-1,
        refit=True,
        return_train_score=True
    )

    search.fit(X_train, y_train)

    best_models[name] = search.best_estimator_

    elapsed = time() - start

    print("\n✅ Finished")

    print("\n📌 Best Params:")
    print(search.best_params_)

    print("\n📈 Best CV Score:")
    print(search.best_score_)

    print(f"\n⏱ Training Time: {elapsed/60:.2f} min")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Data Preparation ---
models = ['Random Forest', 'XGBoost', 'LightGBM', 'Ridge Reg.', 'Linear Reg.']
r2_scores = [0.983, 0.980, 0.980, 0.755, 0.755]
mae_values = [0.015, 0.022, 0.024, 0.212, 0.212]
rmse_values = [0.006, 0.008, 0.008, 0.121, 0.121]

x = np.arange(len(models))  # Label locations
width = 0.25                # Width of the bars

# Custom hex colors matching the original image
color_r2 = '#5D9CEC'    # Soft blue
color_mae = '#FC6E64'   # Soft coral/red
color_rmse = '#7EA23B'  # Olive green

# --- 2. Plot Creation ---
fig, ax1 = plt.subplots(figsize=(12, 7), dpi=300)

# Primary Y-axis: R² Score
rects1 = ax1.bar(x - width, r2_scores, width, label='R² Score', color=color_r2)
ax1.set_ylabel('R² Score (Accuracy)', fontsize=11)
ax1.set_ylim(0, 1.1)
ax1.set_title('Global Model Performance: Accuracy vs. Error', fontsize=14, pad=15)

# Secondary Y-axis: MAE and RMSE
ax2 = ax1.twinx()
rects2 = ax2.bar(x, mae_values, width, label='MAE', color=color_mae)
rects3 = ax2.bar(x + width, rmse_values, width, label='RMSE', color=color_rmse)
ax2.set_ylabel('Error (MAE/RMSE)', fontsize=11)
ax2.set_ylim(0, 0.25)

# X-axis configuration
ax1.set_xticks(x)
ax1.set_xticklabels(models, fontsize=11)

# --- 3. Value Labels Function ---
def autolabel(rects, ax, offset_y=0.01):
    """Attach a text label above each bar, displaying its height."""
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

# Add text labels to the top of each bar
autolabel(rects1, ax1)
autolabel(rects2, ax2)
autolabel(rects3, ax2)

# --- 4. Legends ---
ax1.legend(loc='upper left', frameon=True)
ax2.legend(loc='upper right', frameon=True)

# Clean up layout and display
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- 1. Create the Dataset ---
models = [
    'Linear Regression',
    'Ridge Regression',
    'Random Forest',
    'XGBoost',
    'LightGBM',
]
fertilizers = [
    'MOP',
    'DAP',
    'UREA1',
    'UREA2',
    'UREA3',
    'Organic',
    'Zinc',
    'Boron',
]

data = [
    [0.664, 0.775, 0.632, 0.583, 0.583, 1.0, 0.801, 1.0],  # Linear Regression
    [0.664, 0.775, 0.632, 0.583, 0.583, 1.0, 0.801, 1.0],  # Ridge Regression
    [0.994, 0.998, 0.966, 0.955, 0.955, 1.0, 0.993, 1.0],  # Random Forest
    [0.985, 0.995, 0.964, 0.953, 0.953, 1.0, 0.993, 1.0],  # XGBoost
    [0.986, 0.993, 0.962, 0.952, 0.952, 1.0, 0.992, 1.0],  # LightGBM
]

df = pd.DataFrame(data, index=models, columns=fertilizers)

# --- 2. Plot Configuration ---
plt.figure(figsize=(14, 7), dpi=300)

# Create heatmap using the exact 'Yellow-Green-Blue' colormap from the image
# vmin is set to 0.55 to neatly frame the lowest value (0.583) without crushing the scale
ax = sns.heatmap(
    df,
    annot=True,
    fmt='.3f',
    cmap='YlGnBu',
    vmin=0.55,
    vmax=1.00,
    cbar_kws={'label': '$R^2$ Score'},
    annot_kws={'size': 11},
)

# --- 3. Styling & Labels ---
# Title styling to match the weight and position of the original
plt.title(
    'Fertilizer Prediction Performance Heatmap ($R^2$)',
    fontsize=14,
    fontweight='bold',
    pad=15,
)
plt.ylabel('Model', fontsize=12)

# Adjust tick label aesthetics
plt.xticks(rotation=0, fontsize=11)
plt.yticks(rotation=0, fontsize=11)

# Clean up layout padding
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Data Preparation ---
crops = ['Maize', 'Rice', 'Wheat']
models = ['Linear Regression', 'Ridge Regression', 'Random Forest', 'XGBoost', 'LightGBM']

# Performance metrics mapped by model
data = {
    'Linear Regression': [0.828, 0.658, 0.802],
    'Ridge Regression':  [0.828, 0.658, 0.802],
    'Random Forest':     [0.985, 0.979, 0.986],
    'XGBoost':           [0.982, 0.977, 0.984],
    'LightGBM':          [0.983, 0.975, 0.983]
}

x = np.arange(len(crops))  # Label locations (groups)
width = 0.15               # Width of individual bars

# Precise hex color palette matched from the original/subsequent aesthetics
colors = {
    'Linear Regression': '#5D9CEC',  # Soft Blue
    'Ridge Regression':  '#FC6E64',  # Soft Coral
    'Random Forest':     '#A0C171',  # Muted Sage Green
    'XGBoost':           '#A8BAD8',  # Pastel Blue-Gray
    'LightGBM':          '#A67470'   # Muted Dusty Rose
}

# --- 2. Plot Configuration ---
fig, ax = plt.subplots(figsize=(13, 7), dpi=300)

# Loop through and plot each model's group of bars
# Centers the clusters cleanly over the X ticks
offsets = np.linspace(-width*2, width*2, len(models))

for model, offset in zip(models, offsets):
    rects = ax.bar(x + offset, data[model], width, label=model, color=colors[model])
    
    # Text labels centered cleanly over each individual bar
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 4),  # 4 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=8.5)

# --- 3. Styling & Labels ---
ax.set_title('Crop-wise Model Performance ($R^2$)', fontsize=14, pad=20)
ax.set_xticks(x)
ax.set_xticklabels(crops, fontsize=12)

# Dynamically adjusted Y-axis bounds to completely accommodate 0.658 comfortably
ax.set_ylim(0.60, 1.05) 
ax.tick_params(axis='y', labelsize=11)

# Legend placed uniformly at the lower right to keep bars visible
ax.legend(loc='lower right', frameon=True, fontsize=10)

# Layout adjustments for clean borders
plt.tight_layout()
plt.show()